# MEIDNet Studio in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ABnano/MEIDNet/blob/main/notebooks/04_MEIDNet_Studio_in_Colab.ipynb)

**MEIDNet Studio** is a web page that lays the seven blocks out side by side: change a rule's window or a target and watch the effect flow through every block. Part 1 starts the Studio inside this Colab session and shows it here. Part 2 drives the **same Studio code** from Python, block by block, so you can script what you clicked.

<p><span style="display:inline-block;border-left:5px solid #2a78d6;background:rgba(42,120,214,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">📊 1 · Data</span> → <span style="display:inline-block;border-left:5px solid #eb6834;background:rgba(235,104,52,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🧠 2 · Model</span> → <span style="display:inline-block;border-left:5px solid #1baf7a;background:rgba(27,175,122,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🧱 3 · Family</span> → <span style="display:inline-block;border-left:5px solid #eda100;background:rgba(237,161,0,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">📏 4 · Rules</span> → <span style="display:inline-block;border-left:5px solid #e87ba4;background:rgba(232,123,164,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🎯 5 · Targets</span> → <span style="display:inline-block;border-left:5px solid #008300;background:rgba(0,131,0,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🔎 6 · Search</span> → <span style="display:inline-block;border-left:5px solid #4a3aa7;background:rgba(74,58,167,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">💎 7 · Candidates</span></p>

**How to use it:** run the cells from top to bottom (Shift+Enter, or *Runtime → Run all*). Cells with fields are forms: change a value, then run that cell and the ones below it again.

<sub>Notebook by Anand Babu. MEIDNet: <i>Multimodal generative AI framework for inverse materials design</i>, npj Computational Materials (2026), <a href="https://doi.org/10.1038/s41524-026-02153-3">doi:10.1038/s41524-026-02153-3</a>. 3D views use chemiscope: G. Fraux, R. K. Cersonsky, M. Ceriotti, <i>Chemiscope: interactive structure-property explorer for materials and molecules</i>, JOSS 5, 2117 (2020).</sub>

In [ ]:
#@title ⚙️ Setup: install MEIDNet and load the helpers (run this cell first)
#@markdown In Colab this installs MEIDNet (PyTorch for CPU is enough), chemiscope and ASE, then defines the helpers that every block uses: charts in the block colours, 3D views and report display. It takes about a minute the first time.
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install torch --index-url https://download.pytorch.org/whl/cpu
    !pip -q install git+https://github.com/ABnano/MEIDNet.git chemiscope ase
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()       # chemiscope's 3D viewer is a notebook widget
    from importlib.metadata import version as _installed
    if "numpy" in sys.modules and sys.modules["numpy"].__version__ != _installed("numpy"):
        print("pip updated numpy: choose Runtime -> Restart session, then run this cell again.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
try:
    import meidnet
except ImportError:
    raise ImportError("MEIDNet is not installed in this Python: "
                      "pip install git+https://github.com/ABnano/MEIDNet.git chemiscope ase") from None

# one colour per block, as in MEIDNet Studio; ink and paper of the charts
BLOCK = {"data": "#2a78d6", "model": "#eb6834", "family": "#1baf7a", "rules": "#eda100",
         "targets": "#e87ba4", "search": "#008300", "candidates": "#4a3aa7"}
INK, INK2, MUTED, GRID, AXIS, PAPER, REST = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb", "#d3d2cb"
for _key, _value in {
        "figure.figsize": (6.4, 3.6), "figure.dpi": 110, "figure.facecolor": PAPER, "savefig.facecolor": PAPER,
        "axes.facecolor": PAPER, "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False,
        "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID,
        "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlecolor": INK, "axes.labelcolor": INK2, "axes.labelsize": 10,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
        "legend.frameon": False, "legend.fontsize": 9, "lines.linewidth": 2,
        "lines.solid_capstyle": "round"}.items():
    try:
        plt.rcParams[_key] = _value
    except (KeyError, ValueError):                    # an older matplotlib: keep its default
        pass


def print_flow(to, *items):
    """The last line of a block: what it hands on to the next block."""
    print(f"-> {to}: " + "; ".join(str(i) for i in items))


# ── charts: one axis each, block colours for the data, plain-word titles and labels ──
def _finish(fig, ax, title, xlabel, ylabel, subtitle=None):
    ax.set_title(title, pad=22 if subtitle else 8)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color=INK2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def _window_text(lo, hi):
    if lo is not None and hi is not None:
        return f"{lo:g} to {hi:g}"
    return f"at least {lo:g}" if lo is not None else f"at most {hi:g}"


def hist_prop(values, label, unit="", colour=BLOCK["data"], target=None, window=None, bins=40, title=None,
              ylabel="count"):
    """Histogram in a block colour, with an optional target line and allowed window (min, max)."""
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").astype(float)
    v = v[np.isfinite(v)]
    if v.empty:
        print(f"(no values to plot for {label})")
        return
    fig, ax = plt.subplots()
    ax.hist(v, bins=bins, color=colour, edgecolor=PAPER, linewidth=0.8)
    if window is not None:
        lo, hi = window
        x0, x1 = ax.get_xlim()
        a, b = (x0 if lo is None else lo), (x1 if hi is None else hi)
        ax.axvspan(a, b, color=colour, alpha=0.14, lw=0, zorder=0, label=f"allowed: {_window_text(lo, hi)}")
        ax.set_xlim(min(x0, a), max(x1, b))
    if target is not None:
        ax.axvline(target, color=INK, lw=1.5, label=f"target: {target:g} {unit}".strip())
    if window is not None or target is not None:
        ax.legend(loc="best")
    unit_text = f" [{unit}]" if unit else ""
    stats = f"{len(v):,} values, median {np.median(v):.3g}{' ' + unit if unit else ''}"
    _finish(fig, ax, title or label, f"{label}{unit_text}", ylabel, subtitle=stats)


def funnel_plot(stages, colour=BLOCK["rules"], title="Compositions still allowed after each rule"):
    """Horizontal bars: how many compositions survive each rule, applied in order (stages: [(label, count)])."""
    labels, counts = [str(s[0]) for s in stages], [int(s[1]) for s in stages]
    fig, ax = plt.subplots(figsize=(6.4, 0.42 * len(stages) + 1.2))
    y = np.arange(len(stages))[::-1]
    ax.barh(y, counts, height=0.6, color=colour, edgecolor=PAPER, linewidth=0.8)
    top = max(counts) or 1
    for yi, c in zip(y, counts):
        ax.text(c + 0.01 * top, yi, f"{c:,}", va="center", ha="left", color=INK2, fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, top * 1.15)
    ax.grid(axis="y", visible=False)
    _finish(fig, ax, title, "compositions still allowed", "")


def parity_plot(true, pred, label, unit="", colour=BLOCK["model"]):
    """Predicted against true values; points on the grey diagonal are perfect predictions."""
    t, p = np.asarray(true, dtype=float), np.asarray(pred, dtype=float)
    mae = float(np.mean(np.abs(p - t)))
    ss = float(np.sum((t - t.mean()) ** 2))
    r2 = 1 - float(np.sum((p - t) ** 2)) / ss if ss > 0 else float("nan")
    lo, hi = float(min(t.min(), p.min())), float(max(t.max(), p.max()))
    pad = 0.05 * ((hi - lo) or 1.0)
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=AXIS, lw=1, zorder=1)
    ax.scatter(t, p, s=28, color=colour, edgecolor=PAPER, linewidth=0.8, zorder=2)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect("equal", adjustable="box")
    u = f" [{unit}]" if unit else ""
    _finish(fig, ax, f"{label}: predicted vs true", f"true {label}{u}", f"predicted {label}{u}",
            subtitle=f"mean error {mae:.3g}{' ' + unit if unit else ''}, R² {r2:.2f}, {len(t):,} materials")


def loss_curves(ckpt, colour=BLOCK["model"]):
    """Training loss and validation error per epoch, from the history saved in a MEIDNet checkpoint."""
    import torch
    saved = torch.load(ckpt, map_location="cpu", weights_only=False)
    history = saved.get("history") if isinstance(saved, dict) else None
    if not history or not history.get("train"):
        print("this checkpoint stores no training history (the published v1 model does not)")
        return
    from matplotlib.ticker import MaxNLocator
    train_log, val_log = history["train"], history.get("val") or []
    dots = dict(marker="o", ms=5, mec=PAPER, mew=1)
    losses = [r["total"] for r in train_log]
    fig, ax = plt.subplots()
    ax.plot([r["epoch"] for r in train_log], losses, color=colour, **(dots if len(train_log) <= 30 else {}))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    wide = min(losses) > 0 and max(losses) / min(losses) > 20          # a log axis only when it helps
    if wide:
        ax.set_yscale("log")
    _finish(fig, ax, "Training loss (lower is better)", "epoch", "loss (log scale)" if wide else "loss",
            subtitle=f"{len(train_log)} epochs, last value {losses[-1]:.3g}")
    units = {p["column"]: p.get("unit", "") for p in saved.get("properties", [])}
    for prop in (val_log[-1]["mae"] if val_log else {}):
        fig, ax = plt.subplots()
        ax.plot([r["epoch"] for r in val_log], [r["mae"][prop] for r in val_log], color=colour,
                **(dots if len(val_log) <= 30 else {}))
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        u = units.get(prop, "")
        _finish(fig, ax, f"Validation error of {prop} (lower is better)", "epoch",
                f"mean absolute error{' [' + u + ']' if u else ''}",
                subtitle=f"last value {val_log[-1]['mae'][prop]:.3g}{' ' + u if u else ''}")


def space_scatter(table, x, y, colour=BLOCK["targets"], target=None, highlight=None,
                  highlight_label="closest to the target", xlabel=None, ylabel=None,
                  title="Every composition, as the model predicts it"):
    """Compositions that pass every rule in the block colour, the rejected ones in grey, the target as a cross."""
    ok = table["passes_all"].astype(bool)
    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    ax.scatter(table.loc[~ok, x], table.loc[~ok, y], s=12, color=REST, lw=0,
               label=f"rejected by a rule ({int((~ok).sum()):,})")
    ax.scatter(table.loc[ok, x], table.loc[ok, y], s=30, color=colour, edgecolor=PAPER, lw=0.8,
               label=f"passes every rule ({int(ok.sum()):,})")
    if highlight is not None and len(highlight):
        ax.scatter(highlight[x], highlight[y], s=90, facecolor="none", edgecolor=INK, lw=1.2,
                   label=f"{highlight_label} ({len(highlight)})")
        first = highlight.iloc[0]
        ax.annotate(first["formula"], (first[x], first[y]), xytext=(7, 7), textcoords="offset points",
                    fontsize=9, color=INK)
    if target is not None:
        tx, ty = target
        if ty is None:
            ax.axvline(tx, color=INK, lw=1.5, label="target")
        else:
            ax.scatter([tx], [ty], marker="+", s=220, color=INK, lw=2, zorder=5, label="target")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0)
    _finish(fig, ax, title, xlabel or x, ylabel or y)


# ── 3D: chemiscope (G. Fraux, R. K. Cersonsky, M. Ceriotti, JOSS 5, 2117, 2020) ──
def _as_structure(item):
    """A pymatgen Structure from a Structure, a chemiscope structure dict or an ase.Atoms."""
    from pymatgen.core import Lattice, Structure
    if isinstance(item, Structure):
        return item
    if isinstance(item, dict):
        xyz = np.column_stack([item["x"], item["y"], item["z"]])
        return Structure(Lattice(np.reshape(item["cell"], (3, 3))), item["names"], xyz, coords_are_cartesian=True)
    from pymatgen.io.ase import AseAtomsAdaptor
    return AseAtomsAdaptor.get_structure(item)


def _num(x):
    try:
        f = float(x)
    except (TypeError, ValueError):
        return 0.0
    return f if np.isfinite(f) else 0.0


def _chemiscope_property(name, value, n):
    prop = dict(value) if isinstance(value, dict) else {"values": value}
    values = list(prop["values"])
    if len(values) != n:
        raise ValueError(f"property '{name}' has {len(values)} values for {n} structures")
    if not all(isinstance(x, str) for x in values):
        values = [_num(x) for x in values]          # one type per property, finite numbers only
    out = {"target": prop.get("target", "structure"), "values": values}
    for key in ("units", "description"):
        if prop.get(key):
            out[key] = str(prop[key])
    return out


def show3d(structures, properties=None, settings=None):
    """
    Structures in 3D next to a map of their properties, with chemiscope (G. Fraux, R. K. Cersonsky,
    M. Ceriotti, JOSS 5, 2117, 2020). Click a point of the map to see its crystal. Without chemiscope
    a small table (formula, a, space group) is printed instead.
    """
    structs = [_as_structure(s) for s in structures]
    if not structs:
        print("(no structures to show)")
        return
    props = {name: _chemiscope_property(name, v, len(structs)) for name, v in (properties or {}).items()}
    if sum(not isinstance(p["values"][0], str) for p in props.values()) < 2:     # the map needs two numbers
        props.setdefault("a", {"target": "structure", "values": [round(s.lattice.a, 4) for s in structs],
                               "units": "Å", "description": "cell edge a"})
        props.setdefault("volume_per_atom", {"target": "structure", "units": "Å^3",
                                             "values": [round(s.volume / len(s), 4) for s in structs],
                                             "description": "cell volume per atom"})
    try:
        import chemiscope
        from pymatgen.io.ase import AseAtomsAdaptor
        atoms = [AseAtomsAdaptor.get_atoms(s) for s in structs]
        display(chemiscope.show(atoms, properties=props, settings=settings))
    except Exception as err:                          # no chemiscope (or no widget support): a small table
        print(f"(3D view not available here: {type(err).__name__}: {err})")
        rows = []
        for s in structs[:12]:
            try:
                group = s.get_space_group_info(symprec=0.1)[0]
            except Exception:
                group = "?"
            rows.append({"formula": s.composition.reduced_formula, "a (Å)": round(s.lattice.a, 3),
                         "space group": group})
        print(pd.DataFrame(rows).to_string(index=False))
        if len(structs) > 12:
            print(f"... and {len(structs) - 12} more")


def show_dataset(dataset):
    """Show a chemiscope dataset made by MEIDNet (meidnet.studio.chemiscope or Studio.chemiscope) with show3d."""
    if not dataset.get("structures"):
        print(dataset.get("note", "nothing to show yet"))
        return
    print(f"{dataset['meta']['name']}: {dataset['meta']['description']}")
    show3d(dataset["structures"], dataset.get("properties"), dataset.get("settings"))


# ── tables ──
def space_table(space):
    """meidnet.designspace.enumerate_space(...) as a table: one row per composition, a column per rule."""
    rules = [r["name"] for r in space["rules"]]
    rows = []
    for r in space["rows"]:
        row = {"formula": r["f"], **{f"site_{g}": e for g, e in r["e"].items()}, "a": r["a"]}
        for name in rules:
            row[name] = r["d"].get(name)
            row[f"ok_{name}"] = bool(r["ok"].get(name, True))
        row["passes_all"] = bool(all(r["ok"].values()))
        row.update({f"pred_{c}": v for c, v in r["p"].items()})
        rows.append(row)
    return pd.DataFrame(rows)


def rule_window(params):
    """The allowed (min, max) of a rule from its parameters, or None for a pass/fail rule."""
    lo, hi = params.get("min", params.get("low")), params.get("max", params.get("high"))
    return None if lo is None and hi is None else (lo, hi)


def measured(rule):
    """(what a rule measures, unit), in plain words, for axis labels; other rules fall back to their title."""
    p = rule["params"]
    known = {"min_distance": ("closest distance between two atoms", "Å"),
             "bond_window": (f"nearest {p.get('to', 'B')}–{p.get('from', 'X')} distance ÷ sum of ionic radii", ""),
             "tolerance_factor": ("tolerance factor t", ""),
             "octahedral_factor": ("octahedral factor μ = r_B / r_X", "")}
    return known.get(rule.get("rule", rule["name"]), (rule["title"], ""))


def funnel_stages(table, rules):
    """[(label, count)]: compositions still allowed after each rule, in the family's order."""
    alive = pd.Series(True, index=table.index)
    stages = [("all compositions", len(table))]
    for rule in rules:
        alive &= table[f"ok_{rule['name']}"].astype(bool)
        stages.append((rule["title"], int(alive.sum())))
    return stages


def rank_by_target(table, target, objectives):
    """Sort compositions by the score the search ranks its candidates with (0 = exactly on target)."""
    from meidnet.generate import objective_distance
    score = np.zeros(len(table))
    for o in objectives:
        o = o if isinstance(o, dict) else o.model_dump()
        loss = o.get("select_loss") or o.get("loss", "l2")
        score += o.get("select_weight", 1.0) * np.array(
            [objective_distance(float(v), float(target[o["property"]]), loss) for v in table[f"pred_{o['property']}"]])
    return table.assign(score=score).sort_values("score")


def candidates_table(candidates):
    """One row per saved candidate: where it was found, its predictions and every rule (✓ + measured value)."""
    rows = []
    for c in candidates:
        c = c if isinstance(c, dict) else c.to_dict()
        row = {"formula": c["formula"], "target": c.get("target_index", 1), "round": c["round"],
               "a (Å)": round(c["lattice_a"], 3)}
        row.update({f"predicted {k}": round(v, 3) for k, v in c["predictions"].items()})
        for r in c["constraint_results"]:
            value = r.get("value")
            shown = f" {value:.3g}" if value is not None and r["name"] != "charge_neutrality" else ""
            row[r["name"]] = ("✓" if r["passed"] else "✗") + shown
        row["warnings"] = "; ".join(c.get("flags") or [])
        rows.append(row)
    return pd.DataFrame(rows)


# ── model helpers ──
def predict_records(lm, records):
    """True and predicted (structure -> properties) values for records read by meidnet.pipeline.check."""
    import torch
    x = torch.tensor(np.stack([r.dense for r in records]), dtype=torch.float32, device=lm.device)
    with torch.no_grad():
        z, _ = lm.model.encode_crystal(x)
        predicted = lm.stats.denormalize_tensor(lm.model.property_decoder(z)).cpu().numpy()
    return np.stack([r.properties for r in records]), predicted


def validation_records(cfg):
    """The materials that training held out (the same split as `meidnet train`), for an honest parity plot."""
    from meidnet.data import split_records
    from meidnet.pipeline import check
    info = check(cfg, write_report=False)
    held_out = info["val_records"] or split_records(info["records"], cfg.data.val_fraction, cfg.training.seed)[1]
    if not held_out:
        print("too few materials for a validation split: showing the training materials instead")
        held_out = info["records"]
    return held_out


def show_report(path, height=900):
    """Show one of MEIDNet's HTML reports (check, training, generation) inside the notebook."""
    import html as _html
    with open(path, encoding="utf-8") as f:
        page = f.read()
    if IN_COLAB:                                      # Colab gives every output its own frame
        display(HTML(page))
    else:                                             # elsewhere: a frame keeps the report's style to itself
        display(HTML(f'<iframe srcdoc="{_html.escape(page)}" style="width:100%;height:{height}px;'
                     f'border:1px solid {GRID};border-radius:8px"></iframe>'))
    print("report file:", os.path.abspath(path))


print(f"MEIDNet {meidnet.__version__} is ready" + (" in Colab" if IN_COLAB else ""))

## Part 1 · Open the Studio
The Studio is a small web server. The next cell downloads Perov-5 (the Data block shows it), fetches the published model, starts the server in the background and waits until it answers.

In [ ]:
import shutil
import subprocess
import time
import urllib.request
!meidnet download-data
from meidnet.cli import published_checkpoint
published_checkpoint()                        # fetch the published model once, before the server needs it
STUDIO_PORT = 8765
if "studio_proc" in globals() and studio_proc.poll() is None:
    print("The Studio is already running")
else:
    command = ["meidnet", "studio", "--no-open", "--port", str(STUDIO_PORT), "--host", "127.0.0.1"]
    if shutil.which("meidnet") is None:       # a checkout without the `meidnet` command: the same, through Python
        command = [sys.executable, "-m", "meidnet.cli"] + command[1:]
    studio_log = open("studio_server.log", "w", encoding="utf-8")
    studio_proc = subprocess.Popen(command, stdout=studio_log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{STUDIO_PORT}/api/state", timeout=5) as answer:
                answer.read()
            print(f"MEIDNet Studio is running on port {STUDIO_PORT} (server log: studio_server.log)")
            break
        except Exception:
            if studio_proc.poll() is not None:
                raise RuntimeError("The Studio stopped:\n" + open("studio_server.log", encoding="utf-8").read())
            time.sleep(1)
    else:
        print("The Studio is still starting: run the next cell in a moment (server log: studio_server.log)")

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_iframe(STUDIO_PORT, height=950)
    output.serve_kernel_port_as_window(STUDIO_PORT, anchor_text="Open MEIDNet Studio in its own browser tab")
else:
    print(f"Open http://127.0.0.1:{STUDIO_PORT}/ in your browser")

**Tips.** If the frame stays blank, use the link below it. Deep links open the page on one block: `output.serve_kernel_port_as_iframe(STUDIO_PORT, path="/?panel=rules", height=950)`; `path="/?explore=space"` opens the 3D explorer of the design space (chemiscope).

## Part 2 · The same Studio, from Python
Every button of the page calls a method of the `Studio` class in `meidnet.studio.server`. Below, a separate `Studio` object living in this notebook goes through the seven blocks, so what you do here does not appear in the page above. `sid` names your session, the way a browser tab does.

In [ ]:
from meidnet.studio.server import Studio
s = Studio(None)                              # the published model, as `meidnet studio` without a config file
sid = "colab-notebook"                        # one session = one browser tab: its upload, model and searches
state = s.state(sid)
print(state["model"]["description"])
print("families:", ", ".join(state["families"]))

<div style="border-left:6px solid #2a78d6;background:rgba(42,120,214,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">📊 Block 1 · Data</h2>
<p style="margin:0">What MEIDNet learns from: a table of materials with their structures and properties.</p>
</div>

The Data block shows the data behind the current model and lets you bring your own: upload a table, map its columns, check it. Here 400 rows of Perov-5 stand in for your upload.

In [ ]:
import base64
summary = s.data(sid)                         # the published model's data (Perov-5), once downloaded
print(summary.get("source") or summary.get("note"))
pd.read_csv("data/perov5/train.csv").head(400).to_csv("example_materials.csv", index=False)
with open("example_materials.csv", "rb") as f:
    upload = s.upload({"session": sid, "files": [{"name": "example_materials.csv",
                                                  "b64": base64.b64encode(f.read()).decode("ascii")}]})
print(f"uploaded {upload['rows']} rows; the Studio suggests {upload['suggest']}")

In [ ]:
#@title Map the columns
properties = "heat_all dir_gap"  #@param {type:"string"}
variant = "halide"               #@param ["halide", "oxide", "chalcogenide", "nitride"]
prototype_tolerance = 0.25       #@param {type:"number"}

In [ ]:
checked = s.check_data({"session": sid, "id_column": upload["suggest"]["id_column"],
                        "cif_column": upload["suggest"]["cif_column"],
                        "properties": [{"column": c} for c in properties.split()],
                        "family": "perovskite_abx3", "variant": variant,
                        "align_to_prototype": True, "prototype_tolerance": prototype_tolerance})
print(f"{checked['kept']} of {checked['rows']} rows usable; skipped: {checked['skipped'] or 'none'}")
records = s.session(sid).check_info["records"]
for j, col in enumerate(properties.split()):
    hist_prop([r.properties[j] for r in records], col, colour=BLOCK["data"], ylabel="materials",
              title=f"{col} in the checked upload")
show_dataset(s.chemiscope(sid, "data"))       # the checked structures in 3D, as the page's explorer shows them
print_flow("Model", f"{checked['kept']} checked materials")

<div style="border-left:6px solid #eb6834;background:rgba(235,104,52,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🧠 Block 2 · Model</h2>
<p style="margin:0">One shared latent space where structures and properties meet.</p>
</div>

Train a model for this session on the checked data. A few epochs only show the mechanics; the published model stays available, and the form decides which model the next blocks use.

In [ ]:
#@title Train on your data
epochs = 5            #@param {type:"integer"}
use_my_model = False  #@param {type:"boolean"}
#@markdown Leave **use_my_model** off to continue with the published model: a model trained for a few epochs is only a demonstration.

In [ ]:
started = s.start_train({"session": sid, "epochs": epochs, "batch_size": 16})
print(started)
job = s.status(sid, "train")
while started.get("ok") and not job["done"]:
    time.sleep(2)
    job = s.status(sid, "train")
    p = job["progress"]
    print(f"  epoch {p['epoch']}/{p['epochs']}  loss {p['loss']:.3f}" if p["loss"] is not None else "  starting ...")
if job["error"]:
    print("training failed:", job["error"])
session = s.session(sid)
if session.lm is not None:
    loss_curves(session.lm.path)
    true, predicted = predict_records(session.lm, validation_records(session.cfg))
    for j, (label, unit) in enumerate(zip(session.lm.stats.labels, session.lm.stats.units)):
        parity_plot(true[:, j], predicted[:, j], label, unit, colour=BLOCK["model"])
    s.select_model(sid, "own" if use_my_model else "published")
print(s.state(sid)["model"]["description"])
print_flow("Family", "your model" if s.state(sid)["model"]["own"] else "the published model")

<div style="border-left:6px solid #1baf7a;background:rgba(27,175,122,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🧱 Block 3 · Family</h2>
<p style="margin:0">The crystal type: where atoms sit and which elements may occupy each site.</p>
</div>

The Family block loads a family and variant and lists every composition it allows (the *design space*), with the properties the current model predicts for each.

In [ ]:
from meidnet.family import load_family
from meidnet.constraints import build_candidate
payload = s.family({"session": sid, "family": "perovskite_abx3", "variant": variant})
print(payload["describe"])
display(pd.DataFrame([{"site group": g, "atoms per cell": len(grp["slots"]), "elements allowed": " ".join(grp["sample"])}
                      for g, grp in payload["groups"].items()]))
space = payload["space"]
fam = load_family(payload["name"], variant=payload["variant"])
first = build_candidate(fam, space["rows"][0]["e"])
print(f"\nOne way to fill the prototype: {first.formula()}")
show3d([first.raw], properties={"formula": [first.formula()]})
print_flow("Rules", f"{space['total']:,} compositions")

<div style="border-left:6px solid #eda100;background:rgba(237,161,0,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">📏 Block 4 · Rules</h2>
<p style="margin:0">Hard checks that every candidate must pass, in order.</p>
</div>

The page draws one chart per rule and lets you drag each window. It re-runs no chemistry for that: every composition already carries the value of every rule, so the page only compares numbers with your window. The same comparison, in pandas:

In [ ]:
table = space_table(space)
for rule in space["rules"]:
    print(f"- {rule['title']}: {rule['text']}")
for rule in space["rules"]:
    window = rule_window(rule["params"])
    if window and table[rule["name"]].notna().any():
        label, unit = measured(rule)
        hist_prop(table[rule["name"]], label, unit, colour=BLOCK["rules"], window=window, title=rule["title"],
                  ylabel="compositions")
funnel_plot(funnel_stages(table, space["rules"]), colour=BLOCK["rules"])

In [ ]:
#@title Try a stricter tolerance-factor window
t_min = 0.80  #@param {type:"slider", min:0.6, max:1.0, step:0.01}
t_max = 1.00  #@param {type:"slider", min:0.9, max:1.2, step:0.01}

In [ ]:
overrides = {"tolerance_factor": {"min": t_min, "max": t_max}}
verdict = s.validate({"session": sid, "generation": {"variant": variant, "overrides": overrides}})
print("the Studio accepts these settings" if verdict["ok"] else f"the Studio refuses them: {verdict['errors']}")
others = [f"ok_{r['name']}" for r in space["rules"] if r["name"] != "tolerance_factor"]
table_search = table.assign(passes_all=table[others].all(axis=1) & table["tolerance_factor"].between(t_min, t_max))
print(f"family window: {int(table['passes_all'].sum()):,} compositions pass every rule; "
      f"your window {t_min} to {t_max}: {int(table_search['passes_all'].sum()):,}")
print_flow("Targets", f"{int(table_search['passes_all'].sum()):,} of {len(table_search):,} compositions pass every rule")

<div style="border-left:6px solid #e87ba4;background:rgba(232,123,164,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🎯 Block 5 · Targets</h2>
<p style="margin:0">What the material should have, and how much each goal matters.</p>
</div>

Rank the allowed compositions by how close their predicted properties come to your targets, with the objectives of the Studio's configuration (the score its search uses).

In [ ]:
#@title Set your targets
band_gap = 2.0    #@param {type:"number"}
enthalpy = -0.10  #@param {type:"number"}

In [ ]:
objectives = s.state(sid)["config"]["generation"]["objectives"]
target = {"dir_gap": band_gap, "heat_all": enthalpy}
allowed = table_search[table_search["passes_all"]]
top10 = rank_by_target(allowed, target, objectives).head(10)
sites = [c for c in table_search.columns if c.startswith("site_")]
display(top10[["formula", *sites, "pred_dir_gap", "pred_heat_all", "score"]].round(3).reset_index(drop=True))
space_scatter(table_search, "pred_dir_gap", "pred_heat_all", colour=BLOCK["targets"], target=(band_gap, enthalpy),
              highlight=top10, xlabel="predicted band gap [eV]", ylabel="predicted formation enthalpy [eV/atom]")
hist_prop(allowed["pred_dir_gap"], "predicted band gap", "eV", colour=BLOCK["targets"], target=band_gap,
          title="Predicted band gap of the allowed compositions", ylabel="compositions")

The page's 3D explorer of the design space is a chemiscope dataset; `Studio.chemiscope` returns it, here shown with chemiscope in the notebook (colours: the family's own windows).

In [ ]:
show_dataset(s.chemiscope(sid, "space", "perovskite_abx3", variant))
best = top10["formula"].iloc[0] if len(top10) else "none"
print_flow("Search", f"target band gap {band_gap} eV and enthalpy {enthalpy} eV/atom (closest allowed: {best})")

<div style="border-left:6px solid #008300;background:rgba(0,131,0,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🔎 Block 6 · Search</h2>
<p style="margin:0">The latent optimisation of the paper: from targets back to structures.</p>
</div>

Start a search in the background and poll its status, as the page does every second. `s.stop(sid)` would end it early (the page's Stop button).

In [ ]:
#@title Search budget
n_candidates = 3  #@param {type:"integer"}
rounds = 3        #@param {type:"integer"}
steps = 300       #@param {type:"integer"}
#@markdown A round moves 24 latent points for *steps* gradient steps; the search stops after *rounds* rounds or once *n_candidates* candidates are saved.

In [ ]:
request = {"variant": variant, "targets": [target], "overrides": overrides,
           "per_target": n_candidates, "population": 24, "rounds": rounds, "steps": steps}
started = s.start_search({"session": sid, "generation": request})
print(started)
seen = 0
job = s.status(sid, "search")
while started.get("ok"):
    job = s.status(sid, "search")
    for c in job["candidates"][seen:]:
        print(f"  found {c['formula']:<10} " + ", ".join(f"{k} {v:.3g}" for k, v in c["predictions"].items()))
    seen = len(job["candidates"])
    if job["done"]:
        break
    time.sleep(1)
if job["error"]:
    print("the search failed:", job["error"])
print_flow("Candidates", f"{seen} candidate(s)")

<div style="border-left:6px solid #4a3aa7;background:rgba(74,58,167,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">💎 Block 7 · Candidates</h2>
<p style="margin:0">What came out, and why each one passed.</p>
</div>

What the Candidates block shows: every candidate with each rule and its measured value, the report of the search, the candidates in 3D, and the `meidnet.yaml` that reruns the search from a terminal.

In [ ]:
display(candidates_table(job["candidates"]))
if job["report"]:
    show_report(os.path.join(s.run_root, job["report"]))
show_dataset(s.chemiscope(sid, "candidates"))
yaml_text = s.export({"session": sid, "generation": request})   # the page's "download YAML" button
print(yaml_text)
print_flow("Next steps", "save the YAML above as meidnet.yaml and run `meidnet generate meidnet.yaml` anywhere")

When you are done, stop the server (the page above stops answering):

In [ ]:
if "studio_proc" in globals() and studio_proc.poll() is None:
    studio_proc.terminate()
    print("MEIDNet Studio stopped")

## Where to go next
* On your computer: `pip install meidnet`, then `meidnet studio` (or `meidnet studio meidnet.yaml` with your own model).
* **01** to **03**: the same seven blocks as plain notebook cells.
* Documentation: https://babu09-meidnet.hf.space/docs/

<sub>Notebook by Anand Babu. MEIDNet: <i>Multimodal generative AI framework for inverse materials design</i>, npj Computational Materials (2026), <a href="https://doi.org/10.1038/s41524-026-02153-3">doi:10.1038/s41524-026-02153-3</a>. 3D views use chemiscope: G. Fraux, R. K. Cersonsky, M. Ceriotti, <i>Chemiscope: interactive structure-property explorer for materials and molecules</i>, JOSS 5, 2117 (2020).</sub>